# AcademicComback, Part 2 — Tabitha and the Front EndNotebook 1 was about a *pipeline* — data going in one end and coming out the other. This one is about an *interface*: a thing a person looks at and clicks on.Same approach as before. Where something is a piece of logic, you'll rebuild it in Python and run it. Where something is genuinely a browser thing — a button, a colour, a mouse moving — you'll render it live in this notebook and then go look at the real code.**Read with Tabitha is the better project to learn the front end from**, and here's why: it has no backend at all. No API, no serverless function, no network calls. Everything happens in the browser. So nothing can hide behind "the server did it" — every single thing that happens is in code you can read.---### What you'll be able to do when you finish- Explain what HTML, CSS and JavaScript each do, and where one ends and the next begins- Read any page on your site and know what every tag is for- Understand your whole CSS theme, including the custom-property system that makes it consistent- Explain how JavaScript finds and changes things on a page- Follow all seven of Tabitha's features in the source- Explain why `escapeHtml` exists and what would happen if you deleted it- Find and fix a real bug in your own streak counter

---## Part 1 — The three languagesEvery web page is three languages doing three different jobs.**HTML** is the *structure*. What things are. "This is a heading, this is a button, this is a paragraph." Nouns.**CSS** is the *presentation*. What things look like. "Headings are purple, buttons have a shadow." Adjectives.**JavaScript** is the *behaviour*. What happens. "When this button is clicked, start the timer." Verbs.The reason they're separate is that you can change any one without touching the others. You can restyle your whole site — every colour, every font — without editing a single line of HTML or JavaScript. Your 608-line `style.css` is exactly that: change it and the site looks completely different while behaving identically.The cell below builds a tiny page using all three, in your actual theme colours, and renders it right here in the notebook. Click the button.

In [ ]:
from IPython.display import HTML, displaydemo = '''<div style="all: initial; font-family: Nunito, sans-serif;">  <!-- HTML: the structure -->  <div class="demo-box">    <h3 class="demo-title">This is a heading</h3>    <p class="demo-text">This is a paragraph. Below is a button.</p>    <button class="demo-btn" id="demo-btn">CLICK ME</button>    <p class="demo-out" id="demo-out"></p>  </div>  <!-- CSS: the appearance -->  <style>    .demo-box   { background:#f6f2ff; border:4px solid #2d2440; border-radius:12px;                  padding:1.2rem; max-width:24rem; font-family:Nunito,sans-serif; }    .demo-title { color:#7a5fc0; margin:0 0 .5rem; }    .demo-text  { color:#2d2440; margin:0 0 1rem; }    .demo-btn   { background:#4caf7d; color:#fff; border:3px solid #2d2440;                  border-radius:8px; padding:.7rem 1.2rem; cursor:pointer;                  font-weight:800; box-shadow:0 4px 0 #2d2440; }    .demo-btn:active { transform:translateY(4px); box-shadow:none; }    .demo-out   { color:#4caf7d; font-weight:800; min-height:1.4rem; margin:.8rem 0 0; }  </style>  <!-- JavaScript: the behaviour -->  <script>    (function () {      var count = 0;      document.getElementById("demo-btn").addEventListener("click", function () {        count = count + 1;        document.getElementById("demo-out").textContent =          "Clicked " + count + " time" + (count === 1 ? "" : "s");      });    })();  </script></div>'''display(HTML(demo))

That is a complete web page in miniature, and it's the same three-part shape as every page on your site.Look at how the three connect to each other:- The HTML says `class="demo-btn"`. The CSS says `.demo-btn { ... }`. The dot means "class". That's the link between structure and appearance.- The HTML says `id="demo-btn"`. The JavaScript says `getElementById("demo-btn")`. That's the link between structure and behaviour.**Classes are for styling, ids are for finding.** A class can be on many elements — you want all your buttons to look the same. An id must be unique on the page — JavaScript needs to find exactly one thing. That's the whole distinction, and it's the convention your codebase follows throughout.

---## Part 2 — HTML: the anatomy of your pagesLet's read a real one. Set up the same `show()` helper from notebook 1.

In [ ]:
from pathlib import PathREPO = Path.cwd().parentprint("Project root:", REPO)def show(path, start_marker=None, end_marker=None, max_lines=60, note=None):    p = REPO / path    if not p.exists():        print(f"!! {p} not found"); return    lines = p.read_text(encoding="utf-8").splitlines()    start, end = 0, len(lines)    if start_marker:        for i, line in enumerate(lines):            if start_marker in line:                start = i; break        else:            print(f"!! couldn't find {start_marker!r} in {path}"); return    if end_marker:        for i in range(start + 1, len(lines)):            if end_marker in lines[i]:                end = i + 1; break    end = min(end, start + max_lines)    header = f"  {path}  (lines {start+1}-{end})  "    print("=" * len(header)); print(header); print("=" * len(header))    if note: print(note + "\n")    for n in range(start, end):        print(f"{n+1:4} | {lines[n]}")    print()show("index.html", max_lines=20, note="The top of your landing page.")

**Every page you have starts with this same skeleton.** Going through it line by line:`<!doctype html>` — tells the browser "this is modern HTML". Without it browsers fall into a compatibility mode that behaves oddly. Always the first line, no exceptions.`<html lang="en">` — wraps everything. `lang="en"` tells screen readers what language to pronounce, and helps search engines.`<head>` — information *about* the page that isn't shown on it. Title, description, fonts, stylesheets.`<meta charset="utf-8">` — which character encoding. UTF-8 handles every language and every emoji. Leave it out and your `—` and `🎙️` turn into mojibake.`<meta name="viewport" content="width=device-width, initial-scale=1">` — **this single line is what makes your site work on phones.** Without it, a phone pretends to be a 980px-wide desktop and shrinks everything to unreadable. Every mobile-friendly site has this line.`<link rel="stylesheet" href="css/style.css">` — pull in your CSS.`<link rel="preconnect" href="https://fonts.googleapis.com">` — a performance hint: "I'm going to need this server shortly, start the handshake now." Saves a few hundred milliseconds.`<body>` — everything the user actually sees.### The tags you actually use| Tag | What it's for ||---|---|| `<div>` | a generic box, for grouping and styling. Means nothing on its own || `<span>` | like `<div>` but sits inline, inside a line of text || `<h1>`–`<h6>` | headings, most to least important. One `<h1>` per page || `<p>` | a paragraph || `<a href="...">` | a link || `<button>` | a clickable button || `<input>` | a form field — text, number, file, and so on || `<textarea>` | a multi-line text box || `<label>` | a caption attached to an input. Improves accessibility || `<ul>` / `<ol>` / `<li>` | unordered list / ordered list / list item || `<section>` / `<article>` / `<header>` / `<footer>` | boxes that *mean* something, for screen readers and search engines || `<svg>` | vector graphics drawn with code — how Tabitha's sprite is made |**`<div>` versus `<section>`** is worth knowing. They look identical. But `<section>` tells a screen reader "this is a distinct part of the page". Using the meaningful tag where one exists is called *semantic HTML*, and it's free accessibility.

In [ ]:
show("transcribe.html", start_marker='<div id="intro-step">', end_marker='</div>', max_lines=14,     note="The file-picker area of your transcription page.")

Three things in that fragment worth explaining:**`<input type="file" ... hidden />`** — the file input is deliberately hidden. Browsers style file inputs in their own ugly way and you can't change it much. So the trick is: hide the real input, show a nice-looking `<div>` instead, and when someone clicks the div, tell the hidden input to open. That's this line in your JavaScript:```javascriptdropzone.addEventListener("click", () => fileInput.click());```A very common pattern, and now you know why it exists.**`tabindex="0"` and `role="button"`** — accessibility. The dropzone is a `<div>`, and divs aren't normally keyboard-reachable. `tabindex="0"` puts it in the tab order; `role="button"` tells a screen reader to announce it as a button. Your code also listens for Enter and Space so it works without a mouse. Small effort, and it's the difference between a site some people can't use and one they can.**`disabled`** on the start button — greyed out until files are chosen. The JavaScript flips it: `startBtn.disabled = files.length === 0;`**`hidden`** on `progress-step` and `done-step` — all three screens exist in the HTML from the start; JavaScript just shows one at a time. That's how a page changes without reloading.

---## Part 3 — CSS: how your theme worksCSS rules are always the same shape:```cssselector {  property: value;}```"Find everything matching *selector*, and set *property* to *value*."### Selectors| Selector | Matches ||---|---|| `p` | every `<p>` || `.console` | everything with `class="console"` || `#reader` | the element with `id="reader"` || `.reader h2` | `<h2>`s *inside* `.reader` || `.btn:hover` | `.btn`, but only while the mouse is over it || `.btn:disabled` | `.btn`, but only when disabled || `.reader:empty::before` | invented content shown when `.reader` is empty |That last one is a neat trick you're already using — it's how the reading pane shows placeholder text before any notes are loaded, with no JavaScript involved.### Custom properties: the reason your site looks consistentThe first thing in your stylesheet is this:

In [ ]:
show("css/style.css", start_marker=":root {", end_marker="}", max_lines=22,     note="Your entire colour and font system, defined once.")

`:root` means "the whole document". Anything starting with `--` is a **custom property** — a named value you can reuse everywhere with `var(--name)`.This is the single most valuable idea in your stylesheet. Instead of typing `#7a5fc0` in forty places, you write `var(--lilac-deep)`. Change the one definition and every purple thing on the site changes at once.Try it. The cell below renders your actual palette — edit a hex code and re-run to see how a theme change propagates.

In [ ]:
from IPython.display import HTML, displayimport recss_text = (REPO / "css/style.css").read_text(encoding="utf-8")root = re.search(r":root\s*\{(.*?)\}", css_text, re.S).group(1)tokens = dict(re.findall(r"--([\w-]+):\s*([^;]+);", root))swatches = ""for name, value in tokens.items():    v = value.strip()    if v.startswith("#"):        swatches += (            f'<div style="display:inline-block;margin:6px;text-align:center;font-family:monospace;font-size:11px">'            f'<div style="width:86px;height:56px;background:{v};border:3px solid #2d2440;border-radius:8px"></div>'            f'<div style="margin-top:4px">--{name}</div><div style="color:#666">{v}</div></div>'        )display(HTML(f'<div style="background:#fff;padding:10px">{swatches}</div>'))print("Fonts:")for name, value in tokens.items():    if "font" in name:        print(f"  --{name}: {value.strip()}")

### The box modelEvery element is a box with four layers, from the inside out:```  +-----------------------------------+  |            margin                 |   space OUTSIDE, pushing others away  |  +-----------------------------+  |  |  |         border              |  |   the visible edge  |  |  +-----------------------+  |  |  |  |  |       padding         |  |  |   space INSIDE, between border and content  |  |  |  +-----------------+  |  |  |  |  |  |  |    content      |  |  |  |   your text  |  |  |  +-----------------+  |  |  |  |  |  +-----------------------+  |  |  |  +-----------------------------+  |  +-----------------------------------+```**Padding is inside the border, margin is outside it.** That one sentence resolves most CSS confusion.Your stylesheet's third line is:```css* { box-sizing: border-box; }````*` means "everything". `border-box` changes what `width` means: by default, `width: 200px` plus padding plus border adds up to *more* than 200px, which is maddening. With `border-box`, width means the total — padding and border included. Practically every stylesheet written today starts with this line, and yours should too.The demo below shows the difference. This is one of those things you have to see once.

In [ ]:
display(HTML('''<div style="font-family:Nunito,sans-serif;background:#fff;padding:14px">  <p style="margin:0 0 8px"><b>Both boxes say width: 200px. Both have 20px padding and a 5px border.</b></p>  <div style="display:inline-block;vertical-align:top;margin-right:28px">    <div style="box-sizing:content-box;width:200px;padding:20px;border:5px solid #e05d6f;background:#ffe9ec">      content-box (the default)    </div>    <p style="font-family:monospace;font-size:12px;color:#e05d6f">actual width = 200+40+10 = <b>250px</b></p>  </div>  <div style="display:inline-block;vertical-align:top">    <div style="box-sizing:border-box;width:200px;padding:20px;border:5px solid #4caf7d;background:#e8f7ef">      border-box (what you use)    </div>    <p style="font-family:monospace;font-size:12px;color:#4caf7d">actual width = <b>200px</b>. As asked.</p>  </div></div>'''))

### Layout: flexbox and gridTwo systems for arranging boxes.**Flexbox** lays things out in one direction — a row or a column. Your button rows use it:```css.preset-row, .timer-btns { display: flex; gap: 0.5rem; flex-wrap: wrap; }```"Put these in a row, 0.5rem apart, and wrap onto a new line if there isn't room."**Grid** lays things out in two directions at once. Your Tabitha page uses it for the two-column layout:

In [ ]:
show("css/style.css", start_marker=".tabitha-layout", end_marker="}", max_lines=14)show("css/style.css", start_marker="@media (max-width: 900px)", max_lines=5,     note="And the media query that collapses it to one column on a phone.")

`grid-template-columns: 22rem 1fr` means "first column exactly 22rem wide, second column takes whatever's left". `1fr` is "one fraction of the remaining space" — a grid-only unit, and a very useful one.**The media query** is responsive design in one concept: *when the screen is narrower than 900px, apply these rules instead.* Here it switches the grid to a single column, so the handheld console stacks above the reading pane rather than squashing beside it. Resize your browser on the live site and watch it happen.### CSS units worth knowing| Unit | Meaning ||---|---|| `px` | pixels — fixed || `rem` | multiples of the root font size (usually 16px). `1.5rem` = 24px. **Prefer this** — it scales if a user raises their font size || `em` | multiples of the *current* element's font size || `%` | percentage of the parent || `vw` / `vh` | percentage of the viewport width / height || `fr` | a fraction of leftover space (grid only) || `clamp(a, b, c)` | never below `a`, never above `c`, otherwise `b` |You use `clamp` on your page titles: `clamp(1rem, 4vw, 1.5rem)` means "scale with the screen width, but never smaller than 1rem or bigger than 1.5rem". Responsive text in one line, no media query needed.

---## Part 4 — Run the real site on your own machineEnough reading. Serve your actual site locally and click around it.**Why a server, rather than just double-clicking the HTML file?** Opening a file directly gives you a `file://` address, and browsers restrict what those can do — some things silently behave differently. A local server gives you `http://`, which behaves like the real thing.Run this cell, open the link, and have a look. `python3 -m http.server` is built into Python — nothing to install.

In [ ]:
import subprocess, sys, time, threadingPORT = 8765def serve():    subprocess.run([sys.executable, "-m", "http.server", str(PORT)],                   cwd=str(REPO),                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)threading.Thread(target=serve, daemon=True).start()time.sleep(1.5)print(f"Serving {REPO}\n")print(f"  Landing page : http://localhost:{PORT}/index.html")print(f"  Tabitha      : http://localhost:{PORT}/tabitha.html")print(f"  Transcribe   : http://localhost:{PORT}/transcribe.html   (see note below)")print("\nThe server stops when you shut down this notebook's kernel.")

**Tabitha will work perfectly.** No backend, no API — everything she does happens in the browser.**Transcription will not work locally, and it's worth knowing exactly why.** Two reasons, both of which you now understand:1. `/api/transcribe` doesn't exist here. `python3 -m http.server` serves files; it doesn't run Netlify Functions. To get those locally you'd need `npx netlify dev`.2. The COOP/COEP headers from notebook 1 aren't being sent, because those come from `netlify.toml`, which this plain server knows nothing about. So `SharedArrayBuffer` is off and the audio engine won't start.Both of these are genuinely useful to have seen. A lot of "it works on the live site but not on my machine" confusion comes from not realising your local server is a much simpler thing than your host.> **Try it yourself.** Open Tabitha, press F12 to open DevTools, and click the Elements tab. Hover over lines of HTML and watch the page highlight. Then click something and edit its CSS live in the Styles panel. Nothing you do there is permanent — refresh and it's back. This is the single best way to learn CSS.

---## Part 5 — JavaScript and the DOMWhen a browser loads your HTML it builds a live tree of objects in memory representing the page. That tree is the **DOM** — the Document Object Model. JavaScript doesn't edit your HTML file; it edits that tree, and the browser redraws.### Finding things```javascriptdocument.getElementById("timer-display")     // the one element with that iddocument.querySelector(".reading-outer")     // first match for a CSS selectordocument.querySelectorAll("button")          // all matches```Your code opens with a shortcut used everywhere:```javascriptconst $ = (id) => document.getElementById(id);```That defines a function named `$` that takes an id and returns the element. Now `$("timer-display")` does the same job in far fewer characters. `$` isn't special in JavaScript — it's just a legal variable name, chosen by convention.The `(id) => ...` shape is an **arrow function**, JavaScript's short way of writing a function. These two are the same thing:```javascriptfunction double(x) { return x * 2; }const double = (x) => x * 2;```Python's equivalent is `lambda x: x * 2`.### Changing things```javascriptel.textContent = "hello";          // set text — SAFE, treats it as plain textel.innerHTML  = "<b>hello</b>";    // set HTML — parses tags. Dangerous with user input (Part 8)el.hidden = true;                  // hide itel.disabled = false;               // enable a buttonel.classList.add("on");            // add a classel.classList.toggle("on", flag);   // add if flag is true, remove if falseel.style.top = "40px";             // set one CSS property directly```### Reacting to things```javascriptelement.addEventListener("click", () => { /* runs on every click */ });```"When *this event* happens to *this element*, run *this function*." Events you use: `click`, `input`, `change`, `keydown`, `mousemove`, `dragover`, `drop`.### The wrapper around every one of your scriptsOpen any of your JS files and the first and last lines are:```javascript(() => {  "use strict";  ...})();```This is an **IIFE** — an Immediately Invoked Function Expression. It defines a function and runs it straight away.Why bother? Because every `const` and `function` inside it is trapped inside. Without the wrapper, they'd all be global, shared across every script on the page. `transcribe.js` and `tabitha.js` both define `$` and both define `escapeHtml` — if they were global and ever loaded on the same page, one would silently overwrite the other. The IIFE gives each file its own private room.`"use strict"` turns on stricter rules — notably, assigning to a variable you never declared becomes an error instead of silently creating a global. Always want this.

---## Part 6 — Tabitha, feature by featureSeven features. Let's go through them.### 6a — The spriteTabitha is an **inline SVG**: a picture drawn with code rather than loaded as an image file.

In [ ]:
show("js/tabitha.js", start_marker="const SPRITE", end_marker="</svg>`;", max_lines=32)

`viewBox="0 0 16 16"` sets up a 16×16 coordinate grid. Each `<rect>` is one pixel: x, y, width, height, colour. It's pixel art, written out by hand.`shape-rendering="crispEdges"` stops the browser smoothing the edges when it scales up — without it, blown up large, your sharp pixel art goes blurry.**Why SVG rather than a PNG?** It's vector, so it stays sharp at any size. It's tiny. It's part of the code, so no extra file to download. And you can restyle it with CSS. For pixel art and icons, this is the right choice.Here's Tabitha, pulled live out of your source file and rendered:

In [ ]:
import refrom IPython.display import HTML, displayjs = (REPO / "js/tabitha.js").read_text(encoding="utf-8")svg = re.search(r"(<svg viewBox.*?</svg>)", js, re.S).group(1)display(HTML(f'<div style="background:#f6f2ff;padding:20px;display:inline-block;border:4px solid #2d2440;border-radius:12px">'             f'<div style="width:160px">{svg}</div></div>'))

### 6b — The markdown rendererThis is the most substantial algorithm in Tabitha. It turns markdown text into HTML — headings, bold, lists, code blocks, links — in about 50 lines, with no library.It works in two passes:1. **Block level** — go through line by line. Is this line a heading? A list item? A quote? A code fence?2. **Inline level** — within each line's text, handle `**bold**`, `` `code` ``, `[links](...)`.Below is a **faithful Python port**. Run it, then compare with the original.

In [ ]:
import redef escape_html(s):    return (s.replace("&", "&amp;").replace("<", "&lt;")             .replace(">", "&gt;").replace('"', "&quot;"))def inline_md(s):    s = escape_html(s)                                              # escape FIRST — see Part 8    s = re.sub(r"`([^`]+)`",             r"<code>\1</code>",     s)    s = re.sub(r"\*\*([^*]+)\*\*",       r"<strong>\1</strong>", s)    s = re.sub(r"(^|[^*])\*([^*]+)\*",   r"\1<em>\2</em>",       s)    s = re.sub(r"\b_([^_]+)_\b",         r"<em>\1</em>",         s)    s = re.sub(r"\[([^\]]+)\]\((https?:[^)]+)\)",               r'<a href="\2" target="_blank" rel="noopener">\1</a>', s)    return sdef render_markdown(md_text):    lines = md_text.replace("\r\n", "\n").split("\n")    html = ""    in_code, code_buf = False, []    list_type, list_buf = None, []    def flush_list():        nonlocal html, list_type, list_buf        if list_type:            html += f"<{list_type}>{''.join(list_buf)}</{list_type}>"            list_buf, list_type = [], None    for line in lines:        if re.match(r"^```", line):            if in_code:                html += f"<pre><code>{escape_html(chr(10).join(code_buf))}</code></pre>"                code_buf, in_code = [], False            else:                flush_list(); in_code = True            continue        if in_code:            code_buf.append(line); continue        h = re.match(r"^(#{1,6})\s+(.*)$", line)        if h:            flush_list()            lvl = len(h.group(1))            html += f"<h{lvl}>{inline_md(h.group(2))}</h{lvl}>"            continue        if re.match(r"^\s*>\s?", line):            flush_list()            quoted = re.sub(r"^\s*>\s?", "", line)     # kept out of the f-string:            html += f"<blockquote>{inline_md(quoted)}</blockquote>"   # older Pythons            continue                                                  # reject backslashes in one        ul = re.match(r"^\s*[-*+]\s+(.*)$", line)        ol = re.match(r"^\s*\d+[.)]\s+(.*)$", line)        if ul:            if list_type != "ul": flush_list(); list_type = "ul"            list_buf.append(f"<li>{inline_md(ul.group(1))}</li>"); continue        if ol:            if list_type != "ol": flush_list(); list_type = "ol"            list_buf.append(f"<li>{inline_md(ol.group(1))}</li>"); continue        if re.match(r"^\s*([-*_]){3,}\s*$", line):            flush_list(); html += "<hr>"; continue        if line.strip() == "":            flush_list(); continue        flush_list()        html += f"<p>{inline_md(line)}</p>"    if in_code:        html += f"<pre><code>{escape_html(chr(10).join(code_buf))}</code></pre>"    flush_list()    return htmlSAMPLE = '''# The NephronThe **functional unit** of the kidney. Each kidney has roughly *one million*.## Structure- Renal corpuscle- Proximal convoluted tubule- Loop of Henle1. Filtration2. Reabsorption3. Secretion> Filtration happens under pressure in the glomerulus.Use `eGFR` to estimate function.---See [MDN](https://developer.mozilla.org) for reference.'''out = render_markdown(SAMPLE)print(out[:400], "...\n")

In [ ]:
# And rendered, inside your actual reader styling:from IPython.display import HTML, displaydisplay(HTML(f'''<div style="background:#fffdfa;border:4px solid #2d2440;border-radius:12px;padding:1.4rem;            max-width:40rem;font-family:Nunito,sans-serif;color:#2d2440">  {out}</div>'''))

In [ ]:
show("js/tabitha.js", start_marker="function renderMarkdown", end_marker="return html;", max_lines=45,     note="The JavaScript original. Line for line, the same as the Python above.")

**Two things in this algorithm worth understanding properly.****The `flushList` closure.** List items arrive one line at a time, but `<ul>` has to wrap all of them. So the code collects items in `listBuf` and only writes the `<ul>...</ul>` when the list ends — which is when something that *isn't* a list item shows up. `flushList` is called from many places, and every one of them means "if a list was in progress, finish it now". This pattern — buffer up, flush on change — appears constantly once you start noticing it.**Order matters in `inlineMd`.** `**bold**` is handled before `*italic*`. If you swapped them, the italic rule would chew into the bold markers and produce nonsense. Regular expressions applied in sequence are order-dependent, and that's a classic source of baffling bugs.> **Try it yourself.** Add support for `~~strikethrough~~` producing `<del>`. Where in the sequence does it need to go, and why?

### 6c — Bionic readingBold the first ~40% of each word. The eye latches onto the bold part and the brain fills in the rest, which some people find faster to read.Mechanically this is more delicate than it looks, because you must bold parts of words **without breaking the HTML structure already there**. You can't do a simple find-and-replace on the HTML — you'd corrupt tags and attributes. You have to walk the tree and modify only the *text*, leaving elements alone.That's what `document.createTreeWalker` does: visit every text node, skipping the ones inside `<code>`, `<pre>` and `<a>` (bolding inside a code block or a link would be wrong).Here's the per-word logic in Python:

In [ ]:
import math, redef bionic_word(part):    '''Split one word into (bold_part, rest), matching the JS exactly.'''    m = re.match(r"^(\W*)(\w+)(.*)$", part)    if not m:        return None    pre, word, post = m.groups()    n = max(1, math.ceil(len(word) * 0.4))       # 40%, always at least 1 char    return pre, word[:n], word[n:] + postfor w in ["nephron", "glomerulus", "a", "of", "Bowman's", "(filtration)", "kidney."]:    r = bionic_word(w)    if r:        pre, bold, rest = r        print(f"  {w:15} ->  {pre}[{bold}]{rest}")

In [ ]:
# Render it the way the browser would:from IPython.display import HTML, displaytext = "The functional unit of the kidney is the nephron, and filtration happens in the glomerulus."html_out = ""for part in re.split(r"(\s+)", text):    if not part.strip():        html_out += part; continue    r = bionic_word(part)    if r is None:        html_out += part; continue    pre, bold, rest = r    html_out += f"{pre}<b style='font-weight:800'>{bold}</b>{rest}"display(HTML(f'''<div style="font-family:Nunito,sans-serif;font-size:1.1rem;line-height:1.8;color:#2d2440;            background:#fffdfa;border:4px solid #2d2440;border-radius:12px;padding:1.2rem;max-width:38rem">  <div style="font-size:.75rem;color:#7a5fc0;font-weight:800;margin-bottom:.6rem">BIONIC OFF</div>  <div style="margin-bottom:1.2rem">{text}</div>  <div style="font-size:.75rem;color:#7a5fc0;font-weight:800;margin-bottom:.6rem">BIONIC ON</div>  <div>{html_out}</div></div>'''))

In [ ]:
show("js/tabitha.js", start_marker="function applyBionic", end_marker="node.parentNode.replaceChild", max_lines=32,     note="The tree walker. Note the acceptNode filter skipping CODE, PRE and A.")

`document.createDocumentFragment()` is a performance detail worth knowing. Each change you make to the live page can cause the browser to recalculate layout. Doing that once per word on a long document would be slow. A fragment is an off-screen holding area: build everything in it, then insert it in one go. One layout recalculation instead of hundreds.### 6d — Reading timeThe simplest feature in the app, and a good reminder that useful doesn't mean complicated.

In [ ]:
def read_time(text, wpm=200):    words = len(text.strip().split()) if text.strip() else 0    if not words:        return None    return max(1, round(words / wpm)), wordsfor sample in ["Short note.", SAMPLE, SAMPLE * 12]:    r = read_time(sample)    print(f"  {r[1]:>5} words  ->  ~{r[0]} min")

200 words per minute is the standard estimate for adult silent reading of ordinary prose. Dense technical material is slower — 100–150 is more honest for a pathology lecture. That's a single number in your code you could tune.`Math.max(1, ...)` stops it ever saying "~0 min", which would look broken.### 6e — The Pomodoro timerA **state machine**: a thing that is in exactly one of a few states and moves between them on defined events.```        ┌──────────────── switchMode() ◄──────────────┐        │                                             │        ▼                                             │   ┌─────────┐   remaining hits 0    ┌─────────┐      │   │  FOCUS  │ ────────────────────► │  BREAK  │ ─────┘   │ 25 min  │                       │  5 min  │   └─────────┘                       └─────────┘        ▲                                 │        └──── START / PAUSE / RESET ──────┘```The state is four variables: `mode`, `remaining`, `workMin`, `breakMin`. And `ticker`, which holds the running interval.

In [ ]:
show("js/tabitha.js", start_marker="function stopTimer", end_marker="Back to it", max_lines=26,     note="The whole timer engine: stop, start, and the switch between focus and break.")

**`setInterval(fn, 1000)`** runs a function every 1000 milliseconds and hands back an id. `clearInterval(id)` stops it. Keeping that id in `ticker` is what makes pause possible — without it you could never stop the timer.**The guard in `startTimer`:**```javascriptif (ticker) return;```Without this, clicking START twice would create a *second* interval, and the timer would count down twice as fast. Very easy bug to write, surprisingly hard to diagnose. Any time you start something repeating, guard against starting it twice.**`String(m).padStart(2, "0")`** turns `5` into `"05"`. Python's equivalent is `f"{m:02d}"`.### 6f — The study streak, and a real bugThe streak stores the last study date and a count. Logic: same day, leave it; next day, increment; longer gap, reset to 1.Straightforward — except for one thing.

In [ ]:
show("js/tabitha.js", start_marker="function todayStr", end_marker="$(\"streak\").textContent", max_lines=26)

Look closely at:```javascriptfunction todayStr() { return new Date().toISOString().slice(0, 10); }```**`toISOString()` always returns UTC.** Nigeria is UTC+1. So between midnight and 1am local time, this function returns *yesterday's* date.That's not just cosmetic. Run the cell below.

In [ ]:
from datetime import datetime, timezone, timedeltaWAT = timezone(timedelta(hours=1))                    # Nigeriadef js_today(local_dt):    '''Exactly what new Date().toISOString().slice(0,10) gives you.'''    return local_dt.astimezone(timezone.utc).date().isoformat()def days_between(a, b):    return (datetime.fromisoformat(b) - datetime.fromisoformat(a)).daysa = datetime(2026, 9, 22,  0, 30, tzinfo=WAT)          # Tuesday 00:30, studying lateb = datetime(2026, 9, 22, 23,  0, tzinfo=WAT)          # Tuesday 23:00, same dayprint("Two study sessions on the SAME local Tuesday:\n")print(f"  00:30 local  ->  stored as {js_today(a)}")print(f"  23:00 local  ->  stored as {js_today(b)}")print(f"\n  daysBetween = {days_between(js_today(a), js_today(b))}")print("  the code sees a gap of 1 day and INCREMENTS the streak")print("  -> one day of studying counted as two\n")print("Using the LOCAL date instead:")print(f"  00:30 local  ->  {a.date()}")print(f"  23:00 local  ->  {b.date()}")print(f"  daysBetween = {(b.date() - a.date()).days}   -> no increment. Correct.")

**So Tabitha's streak can be inflated by studying late at night and then again the next evening.** It's a small bug in a fun feature, not a disaster — but it's a genuine one, in your code, found by reasoning about it rather than by it breaking.The fix is to build the date from local parts instead of the UTC string:```javascriptfunction todayStr() {  const d = new Date();  const pad = (n) => String(n).padStart(2, "0");  return `${d.getFullYear()}-${pad(d.getMonth() + 1)}-${pad(d.getDate())}`;}````getMonth()` returns 0 for January, which is why it needs `+ 1`. A famous JavaScript trap — `getDate()` is 1-based but `getMonth()` is 0-based.> **Try it yourself.** Make that change in `js/tabitha.js`. Then work out what else breaks: anyone with an existing streak has a UTC date stored in their browser. Does your fix handle that gracefully, or does it reset their streak?### 6g — The reading guideA horizontal bar that follows your mouse to help you keep your place.

In [ ]:
show("js/tabitha.js", start_marker="readingWrap.addEventListener(\"mousemove\"", max_lines=8)

`getBoundingClientRect()` gives an element's position and size **relative to the browser window**. The mouse event's `clientY` is also relative to the window. Subtract one from the other and you get the mouse position *inside* that element — which is what you need to place the bar.Then `- guideBar.offsetHeight / 2` centres the bar on the cursor rather than hanging it below.This is the standard way to convert "where is the mouse on screen" into "where is the mouse inside this box", and you'll need it for any drag, draw or hover-position feature you ever build.### 6h — The beep`switchMode()` calls `beep()`, which uses the **Web Audio API** to synthesise a tone from nothing — no sound file needed.

In [ ]:
show("js/tabitha.js", start_marker="function beep()", end_marker="} catch", max_lines=14)

An **oscillator** generates a wave at a frequency (660 Hz here, an E note). A **gain** node controls volume. You connect them in a chain — oscillator into gain, gain into the speakers — then fade the gain down exponentially over half a second so it doesn't end with an ugly click.The whole thing is wrapped in `try`/`catch` because browsers block audio until the user has interacted with the page. Since the timer only starts after a click, you're fine — but if it ever fails, silence is a perfectly acceptable outcome, so the catch does nothing. **A `catch` that deliberately does nothing is fine, as long as you say so in a comment** — which yours does.

---## Part 7 — localStorage: remembering thingsBoth the name gate and the streak use `localStorage`, a small store the browser keeps per site.```javascriptlocalStorage.setItem("key", "value");     // savelocalStorage.getItem("key");              // read (null if absent)localStorage.removeItem("key");           // delete```Four things to know about it:1. **It only stores strings.** To keep an object you `JSON.stringify` on the way in and `JSON.parse` on the way out. That's why your code does exactly that.2. **It's per-browser, per-device.** A student's streak on their phone is a completely different streak from their laptop. There's no sync without a real backend.3. **It can throw.** In private browsing, or with site data blocked, `setItem` raises. Your code wraps every access in `try`/`catch` — correct, and often forgotten.4. **It is not secure.** The user can read and edit it in DevTools. Fine for a streak counter. Never for anything that matters.

In [ ]:
show("js/namegate.js", start_marker="const STORAGE_KEY", end_marker="window.ACT_getUser", max_lines=16,     note="Your name gate's storage layer. Note the try/catch on both read and write.")

The name gate's structure is worth a look as a piece of design:```javascriptif (getUser()) return;```If a name is already stored, the script stops immediately and does nothing else. The overlay is only ever built for a first-time visitor. Cheap, and it means returning visitors pay no cost at all.Then the overlay is created entirely in JavaScript — HTML *and* its CSS — and injected. That's why it works on every page with one `<script>` tag and no markup changes. A reasonable trade-off for something this self-contained, though for anything larger you'd want the styles in your stylesheet where you can find them.Note also `registeredAt: new Date().toISOString()` and the comment about adding an email later. Storing an object rather than a bare string means you can add fields later without breaking data already sitting in people's browsers. Small decision, saves real pain.

---## Part 8 — Why `escapeHtml` existsThis is the most important security concept in front-end work, and your code handles it correctly — so let's make sure you know *why*, and don't accidentally undo it later.**The problem.** `innerHTML` doesn't just insert text — it *parses* it as HTML. So if a user's input reaches `innerHTML` unescaped, whatever tags they typed become real elements on the page. Including `<script>`.That's **XSS** — cross-site scripting. On a site with accounts it's how attackers steal sessions. Tabitha has no accounts and the notes never leave the browser, so the real-world risk here is low — but the habit is what matters, because the next thing you build might have accounts.**The fix** is to convert the dangerous characters into their harmless display equivalents before they're parsed:

In [ ]:
def escape_html(s):    return (s.replace("&", "&amp;").replace("<", "&lt;")             .replace(">", "&gt;").replace('"', "&quot;"))hostile = '<img src=x onerror="alert(\'your session is mine\')">'print("Raw, straight into innerHTML:")print("   ", hostile)print("    -> browser builds a real <img>, it fails to load, onerror fires, attacker's code runs\n")print("Escaped first:")print("   ", escape_html(hostile))print("    -> browser displays those characters as text. Nothing executes.")

**`&` has to be replaced first.** If you did `<` before `&`, you'd turn `<` into `&lt;` and then that new `&` would itself become `&amp;lt;`, giving you visible garbage. Order matters. Your JS does it in one pass with a lookup table, which sidesteps the problem entirely — a neater solution:```javascriptreturn s.replace(/[&<>"]/g, (c) => ({ "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;" }[c]));```**And note where it's called** in `inlineMd`: escaping happens *first*, before any markdown conversion. That ordering is essential. The `<strong>` tags your own code adds afterwards are trusted, because you wrote them. Anything that came from the user was neutralised before that point.**The rule to carry forward:** `textContent` is always safe. `innerHTML` is safe only when you are certain every part of the string either came from you or has been escaped. When both would work, use `textContent`.> **Try it yourself.** Open Tabitha locally, paste `<img src=x onerror="alert(1)">` into the notes box, and load it. You'll see the text, not an alert. Then temporarily delete the `escapeHtml` call from `inlineMd` and try again. Put it back afterwards.

---## Part 9 — Rebuild it yourself### Level 1 — read the page1. Open Tabitha in your browser with DevTools open. Using only the Elements panel, find the element with `id="reader"`. What are its parents, all the way up?2. In the Styles panel, change `--lilac-deep` on `:root`. Watch the whole page change.3. Find where `.btn-mini.on` is defined. Now click BIONIC and watch that class get added live.### Level 2 — build from scratchMake a new folder with one HTML file, one CSS file and one JS file, and build a page with a text box and a button that shows the word count of whatever's typed. No copying from your project, no AI. You'll need: `<textarea>`, `<button>`, `getElementById`, `addEventListener`, `textContent`.Then, once it works, add:1. A live count that updates as you type (the `input` event, not `click`)2. A reading-time estimate3. A button that toggles a dark mode (`classList.toggle`)4. Remember the dark-mode choice in `localStorage`That is genuinely a real web app, and it's four evenings of work at most.### Level 3 — change Tabitha1. **Fix the streak bug** from Part 6f.2. Add a `~~strikethrough~~` rule to the markdown renderer.3. Make the bionic percentage adjustable — a slider from 20% to 60%.4. Add a "copy to clipboard" button for the rendered notes (`navigator.clipboard.writeText`).5. Save the notes to `localStorage` so a refresh doesn't lose them. Think about what happens when the notes are very large.### Level 4 — explain itExplain to someone non-technical what happens, in order, from clicking LOAD NOTES to seeing formatted text. Every step. If you have to hand-wave, that's your next thing to study.

---## Part 10 — Reference card### DOM```javascriptdocument.getElementById("x")        document.querySelector(".x")document.querySelectorAll("button") el.textContent = "safe"el.innerHTML = "<b>careful</b>"     el.classList.add/remove/toggle("x")el.hidden = true                    el.disabled = falseel.style.top = "40px"               el.getBoundingClientRect()document.createElement("div")       parent.appendChild(child)el.remove()                         el.dataset.work   // reads data-work="25"```### Events```javascriptel.addEventListener("click",  (e) => {})el.addEventListener("input",  (e) => {})   // fires on every keystrokeel.addEventListener("change", (e) => {})   // fires when done (files, selects)el.addEventListener("keydown",(e) => { if (e.key === "Enter") {} })e.preventDefault()        // stop the browser's default behavioure.target                  // what was actually clickede.currentTarget           // what the listener is attached toe.target.closest("button")// nearest matching ancestor```### Timing```javascriptconst id = setInterval(fn, 1000);  clearInterval(id);const id = setTimeout(fn, 3000);   clearTimeout(id);```### Storage```javascriptlocalStorage.setItem("k", JSON.stringify(obj));JSON.parse(localStorage.getItem("k") || "null");```### CSS selectors```css.class    #id    tag    .a .b (descendant)    .a > .b (direct child).a:hover  .a:focus  .a:disabled  .a:empty  .a::before  .a:first-child@media (max-width: 900px) { }```### CSS layout```cssdisplay: flex; gap: .5rem; flex-wrap: wrap;justify-content: center;   /* along the main axis  */align-items: center;       /* across the main axis */display: grid; grid-template-columns: 22rem 1fr;```### JavaScript vs Python| JavaScript | Python ||---|---|| `const` / `let` | just assign || `(x) => x * 2` | `lambda x: x * 2` || `null` / `undefined` | `None` || `===` | `==` || `&&` `\|\|` `!` | `and` `or` `not` || `arr.map(f)` | `[f(x) for x in arr]` || `arr.filter(f)` | `[x for x in arr if f(x)]` || `` `hi ${name}` `` | `f"hi {name}"` || `try { } catch (e) { }` | `try: / except as e:` || `JSON.stringify` / `JSON.parse` | `json.dumps` / `json.loads` || `Math.max` `Math.floor` | `max` `math.floor` || `str.padStart(2,"0")` | `f"{n:02d}"` |### Look it up here, not from an AI- MDN — <https://developer.mozilla.org> — the reference for all three languages- CSS Tricks flexbox guide — <https://css-tricks.com/snippets/css/a-guide-to-flexbox/>- CSS Tricks grid guide — <https://css-tricks.com/snippets/css/complete-guide-grid/>---**Next:** notebook 3 — git, GitHub, Netlify, environment variables, and what actually happens between `git push` and your site updating.